In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('../src')
import pandas as pd
from pre_processing import df_transform_experimental, prepare_for_training
from training import run_cv_training
from analysis import *
from custom_scoring import calculate_macro_ap
import os
os.environ["WANDB_SILENT"] = "true"
import wandb

DATA_PATH = '../data/train.csv'
CLASS_NAMES = ["Clutter", "Cormorants", "Pigeons", "Ducks", "Geese", "Gulls", "Birds of Prey", "Waders", "Songbirds"]

params = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 6,
            'objective': 'multi:softprob',
            'num_class': len(CLASS_NAMES),
            'tree_method': 'hist',
            'early_stopping_rounds': 50,
            'random_state': 42,
            'use_weights':False,
        }


df = pd.read_csv(DATA_PATH)
transformed = df_transform_experimental(df, True)
X, y, features = prepare_for_training(transformed)

config = {
    'params':params,
    'feature_version':features,
}

In [ ]:
wandb.init(project="AICup", config=config)
models, oof_preds= run_cv_training(X, y, class_names=CLASS_NAMES, params=params)
final_score, detailed_scores = calculate_macro_ap(y, oof_preds, CLASS_NAMES)
wandb.log({"mean_macro_ap": final_score})
wandb.log({"detailed_scores": detailed_scores})
wandb.finish()

In [ ]:
print_detailed_scores(final_score, detailed_scores)
debug_per_class_scores(y, oof_preds)

In [ ]:
plot_multiclass_calibration(y, oof_preds)